# DB-PatchKAN — Kaggle T4 smoke and reproduction

Uses FP16, gradient accumulation, and five epochs per stage to validate hierarchical trip pooling, memory use, checkpointing, and one-fold reproducibility on a 16-GB T4. SSL uses 32 independent segments per batch; supervised KAN/MLP runs preserve every segment in one trip bag at a time.

In [ ]:
import os
import subprocess
from pathlib import Path
import torch

REPO = Path('/kaggle/working/dbpatchkan')
os.chdir(REPO)
subprocess.run(['pip', 'install', '-q', '-e', '.'], check=True)
assert torch.cuda.is_available(), 'Enable the Kaggle GPU accelerator.'
print(torch.cuda.get_device_name(0))

In [ ]:
DATA = Path('/kaggle/input/dbpatchkan-segments/segments_train.npz')
VALIDATION = Path('/kaggle/input/dbpatchkan-segments/segments_validation.npz')
OUTPUT = Path('/kaggle/working/dbpatchkan_t4_smoke')
assert DATA.exists() and VALIDATION.exists(), 'Attach the preprocessed segment dataset.'

command = [
    'python', '-m', 'dbpatchkan.training.train',
    '--data', str(DATA), '--validation-data', str(VALIDATION),
    '--trip-bags',
    '--output-dir', str(OUTPUT), '--stage', 'both',
    '--device', 'cuda', '--precision', 'fp16', '--domain', 'nissan',
    '--segment-length', '256', '--patch-length', '16', '--patch-stride', '8',
    '--d-model', '128', '--encoder-layers', '3', '--heads', '4', '--ff-dim', '256',
    '--dropout', '0.10', '--kan-grid', '5', '--fusion-head', 'kan',
    '--pretrain-batch-size', '32', '--finetune-batch-size', '1',
    '--gradient-accumulation', '4', '--pretrain-epochs', '5', '--epochs', '5',
    '--pretrain-lr', '3e-4', '--encoder-lr', '1e-5', '--head-lr', '3e-4',
    '--weight-decay', '0.01', '--warmup-fraction', '0.05',
    '--gradient-clip', '1.0', '--mask-ratio', '0.4', '--patience', '5',
    '--num-workers', '4', '--seed', '42', '--freeze-encoders',
]
subprocess.run(command, check=True)

## Parameter-matched MLP challenger
The identical data and training configuration is rerun with only the concept fusion head changed.

In [ ]:
mlp_command = command.copy()
mlp_command[mlp_command.index('kan')] = 'mlp'
mlp_command[mlp_command.index('--output-dir') + 1] = str(OUTPUT.parent / 'dbpatchkan_t4_mlp')
subprocess.run(mlp_command, check=True)

In [ ]:
import json
print(json.loads((OUTPUT / 'history.json').read_text())[-1])
print('Peak allocated GB:', torch.cuda.max_memory_allocated() / 2**30)